# 5.3 From notebook to script

Every notebook so far has been the whole story: cells run once, top to bottom, and the
narrative *is* the code. That stops working the moment you want to run the same analysis
twice — on tomorrow's data, on a schedule, from a `Makefile` — because a cell has no natural
boundary. Nothing stops one cell's variable from leaking into the next, and nothing forces
the logic small enough to test or reuse.

This notebook is about the *shape* of an analysis once it leaves the notebook, and it borrows
its content from [04.2](../lesson4/04.2-distribution_fitting.ipynb) on purpose: the covid
story has been told, so what is left to look at is the structure.
`scripts/covid_pipeline.py` is that story as a script — config, process, compare, model,
residual, distribution fit — with a `main()` that runs the whole loop and six functions that
each do one thing. Watch for how little of what follows is about COVID.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.visualizer import ComparePlot, PlotSettings

from scripts.covid_pipeline import (
    fit_model,
    plot_model,
    plot_residual,
    plot_residual_distribution,
    plot_zscores,
    preprocess,
)
from wa_analyzer.data import load_showcase

## The loop, as six functions

Open `scripts/covid_pipeline.py` next to this notebook. Each function has a name a reader can
guess the contents of, and a signature that says what goes in and what comes out:

| function | in | out | the decision it owns |
|---|---|---|---|
| `preprocess()` | nothing (config) | the processed frame | which pipeline steps make the data |
| `plot_zscores(data)` | the frame | a figure | whether the two series share a shape |
| `fit_model(data)` | the frame | the frame, plus `predicted deaths` and `residual` | the model, its initial values, its bounds |
| `plot_model(data)` | the fitted frame | a figure | what the fit looks like against the data |
| `plot_residual(data)` | the fitted frame | a figure | whether the error has structure over time |
| `plot_residual_distribution(data)` | the fitted frame | a figure and a `fit_table` | whether the error is noise |

None of them is more than twenty lines, and none of them knows about the others: `fit_model`
returns a *new* frame rather than editing the one it was given, so running it twice does not
double-fit, and every plotting function takes the frame it needs as an argument rather than
reaching for a global. That is the entire trick — not that the lines are different from the
notebook's, but that they are *separate*, in a file, behind names.

Run the loop, one function per cell, in the order `main()` runs it.

In [ ]:
data = preprocess()
fig, ax = plot_zscores(data)

Deaths, shifted back by the reporting lag, against positive tests — both on one scale. They
track each other through the winter and part company in the spring, which is the shape 04.2
built its model around.

In [ ]:
data = fit_model(data)
fig, ax = plot_model(data)

In [ ]:
fig, ax = plot_residual(data)

In [ ]:
fig, table = plot_residual_distribution(data)
table[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(3)

The same four pictures 04.2 ended on — the model whose ratio turns, a residual without a
trend, a residual distribution several families fit equally well — produced by six function
calls instead of thirty cells. The log lines above them are the script talking: what it
loaded, what it fitted, where it saved each figure. That is what a script gives you that a
notebook does not: a record of a run, and a run that can happen without you.

## Why the functions are the point

Three things you can now do that you could not with the notebook version:

1. **Re-run one step.** Change the model in `fit_model` and re-run it; nothing upstream has
   to execute again, and nothing downstream can have used a stale variable.
2. **Import it elsewhere.** This notebook did. A dashboard could. A test could call
   `fit_model` on a frame of ten made-up rows and check the residual has mean zero.
3. **Run it on a schedule.** `uv run python scripts/covid_pipeline.py` needs no kernel, no
   cell order, and no human.

The cost is small and worth naming: a function has to *say* what it needs. `plot_residual`
takes `data`; it cannot quietly use the `data` from three cells up. That constraint is the
feature.

## One more basis function

04.2's fix was a logistic switch multiplied into a straight line. That is one of a small family
of shapes worth knowing by name — **linear, sine, exponential, logistic** — and the habit is
the same for each: look at what the current model cannot explain, name the shape of what is
left, and add that shape as a term. Monthly airline passengers, 1949–1960, is the plainest
case: a trend, and a cycle that repeats every twelve months.

In [ ]:
flights = load_showcase("flights")
months = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
t = np.arange(len(flights)).astype(float)
y = flights["passengers"].to_numpy().astype(float)


def trend_plus_seasonal(t: np.ndarray, params: list[float]) -> np.ndarray:
    a, b, amplitude, phase = params
    return a * t + b + amplitude * np.sin(2 * np.pi * t / 12 + phase)


trend_only = train_model(t, y, linear_model, mse, [1.0, 100.0])
trend_and_cycle = train_model(t, y, trend_plus_seasonal, mse, [1.0, 100.0, 30.0, 0.0])
compare = pd.DataFrame({
    "month": months,
    "actual": y,
    "trend only": linear_model(t, trend_only),
    "trend + 12-month sine": trend_plus_seasonal(t, trend_and_cycle),
})
for column in ["trend only", "trend + 12-month sine"]:
    explained = 1 - np.var(y - compare[column]) / np.var(y)
    print(f"{column:22s} variance explained {explained:.1%}")

settings = PlotSettings(figsize=(10, 4.5), title="Airline passengers: a line, and a line plus a sine",  # ty: ignore[invalid-argument-type]
                        xlabel="", ylabel="passengers")
fig, ax = ComparePlot(settings).plot(data=compare, x="month", y1="trend only", y2="trend + 12-month sine")
ax.scatter(compare["month"], compare["actual"], s=8, color="black", alpha=0.5, label="actual", zorder=3)
_ = ax.legend()

One added term — the amplitude and phase of a 12-month sine, four parameters in total — lifts
the explained variance from 85% to 93%. The gap that remains is visible and honest: the real
cycle *widens* as the trend grows, and a sine added on top of a line stays the same width
throughout. Getting the rest would mean multiplying the cycle by the trend instead of adding it
— a different, still simple, basis function, and the natural next term rather than a failure
of this one.

## What to carry forward

1. **A script is a notebook with the leakage removed.** Small functions, each independently
   callable, are what let you re-run one step, import it elsewhere, or test it without
   re-running everything above it.
2. **The residual is the finding, not the leftover.** Every model above was improved by
   asking what the *error* still looked like, not by staring harder at the fit.
3. **A basis function is a claim about the shape you expect.** A logistic said "the ratio
   turns". A sine said "this repeats every twelve months". Both are testable, and both were
   confirmed by the residual shrinking once they were added — not assumed because they seemed
   reasonable.